In [1]:
%%writefile config.py
"""
Configuration file for Dual-Encoder Copy-Move Forgery Detection Model
Save as: config.py
"""

import torch

class Config:
    # ==================== PATHS ====================
    TRAIN_AUTHENTIC_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/synthetic_forgery_dataset/authentic"
    TRAIN_FORGED_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/synthetic_forgery_dataset/forged/images"
    TRAIN_MASK_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/synthetic_forgery_dataset/masks"
    
    TEST_AUTHENTIC_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection/train_images/authentic"
    TEST_FORGED_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection/train_images/forged"
    TEST_MASK_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection/train_masks"
    
    HQ_AUTHENTIC_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection"
    HQ_FORGED_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/supplemental_images"
    HQ_MASK_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/supplemental_masks"
    
    OUTPUT_DIR = "/kaggle/working/outputs"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    
    # ==================== MODEL ====================
    INPUT_SIZE_SWIN = 224
    INPUT_SIZE_CONVNEXT = 448
    INPUT_CHANNELS = 3
    NUM_INSTANCES = 6
    
    # Dual Encoders
    SWIN_MODEL = "swin_small_patch4_window7_224"
    CONVNEXT_MODEL = "convnext_small"
    EMBEDDING_DIM = 128
    
    # Decoder
    DECODER_CHANNELS = [512, 256, 128, 64, 32]
    
    # Fusion - Channel reduction to 2/3 of concatenated
    FUSION_REDUCTION_RATIO = 2/3
    
    # ==================== TRAINING ====================
    NUM_EPOCHS = 50
    BATCH_SIZE = 17
    GRADIENT_ACCUMULATION_STEPS = 2
    ENCODER_FREEZE_EPOCHS = 10
    
    LEARNING_RATE = 5e-4
    WEIGHT_DECAY = 1e-4
    OPTIMIZER = "AdamW"
    
    T_MAX = NUM_EPOCHS
    ETA_MIN = 5e-6
    
    # Mixed Precision Training
    USE_AMP = True
    
    EARLY_STOPPING_PATIENCE = 5  # Stop if no improvement for 8 validations
    
    # ==================== LOSS WEIGHTS ====================
    # Main decoder losses (scaled down for multi-task learning)
    MAIN_DECODER_SCALE = 0.4  # Reduce main decoder influence
    LOSS_WEIGHT_DICE = 2
    LOSS_WEIGHT_SEGMENTATION_FOCAL = 1.5
    LOSS_WEIGHT_ACTIVE = 4.0
    
    # Auxiliary head losses (dominant to make encoders learn their duties)
    LOSS_WEIGHT_SWIN_CONTRASTIVE = 2.0  # Heavy weight for similarity learning
    LOSS_WEIGHT_CONVNEXT_COARSE = 0.8   # Heavy weight for coarse forgery detection
    
    # Focal loss parameters
    FOCAL_ALPHA = 0.85
    FOCAL_GAMMA = 2.0
    SEG_FOCAL_ALPHA = 1
    SEG_FOCAL_GAMMA = 0
    
    # Contrastive Config (for Swin Aux Head)
    CONTRASTIVE_TEMPERATURE = 0.07
    CONTRASTIVE_SAMPLES_BACKGROUND = 256  # More samples from background (larger area)
    CONTRASTIVE_SAMPLES_PER_INSTANCE = 64  # Fewer samples per instance
    CONTRASTIVE_MARGIN_BG_FG = 1.5   
    CONTRASTIVE_MARGIN_FG_FG = 1.0   
    CONTRAST_WEIGHT_BG_FG = 2.0      
    CONTRAST_WEIGHT_FG_FG = 1.2      
    
    # ==================== AUGMENTATION ====================
    AUG_ROTATION_LIMIT = 90 
    AUG_PROB_FLIP = 0.5
    AUG_PROB_ROTATE = 0.65
    
    AUG_BRIGHTNESS = 0.25
    AUG_CONTRAST = 0.15
    AUG_SATURATION = 0.15
    AUG_HUE = 0.05
    AUG_PROB_COLOR = 0.5
    
    AUG_JPEG_QUALITY_RANGE = (90, 100) 
    AUG_PROB_JPEG = 0.35
    
    AUG_BLUR_LIMIT = (3, 6) 
    AUG_PROB_BLUR = 0.35
    
    AUG_NOISE_VAR_LIMIT = (5.0, 15.0) 
    AUG_PROB_NOISE = 0.3
    
    # ==================== INFERENCE ====================
    ACTIVE_CHANNEL_THRESHOLD = 0.5
    MASK_THRESHOLD = 0.5
    
    # ==================== VALIDATION ====================
    VAL_SAMPLE_RATIO = 0.5  # Use 50% of validation data
    VAL_AUTHENTIC_RATIO = 0.3  # 30% authentic in sampled data
    VAL_FORGED_RATIO = 0.7  # 70% forged in sampled data
    VAL_FREQUENCY = 2  # Validate every epoch (changed from 3)
    
    # ==================== THRESHOLD TESTING ====================
    TEST_ACTIVE_THRESHOLDS = [0.25, 0.5, 0.75]
    TEST_MASK_THRESHOLDS = [0.25, 0.5, 0.75]
    
    # ==================== DISTRIBUTED ====================
    WORLD_SIZE = 2
    BACKEND = "nccl"
    
    # ==================== TESTING ====================
    SAVE_TOP_WORST = 100
    
    # ==================== SMOKE TEST ====================
    SMOKE_TEST = False
    SMOKE_TEST_SAMPLES = 30
    SMOKE_TEST_EPOCHS = 30
    SMOKE_TEST_BATCH_SIZE = 15
    SMOKE_TEST_FREEZE_EPOCHS = 10
    SMOKE_TEST_GRAD_ACCUM = 1  # No grad accumulation in smoke test
    
    @classmethod
    def set_smoke_test(cls, enable=True):
        cls.SMOKE_TEST = enable
        if enable:
            cls.NUM_EPOCHS = cls.SMOKE_TEST_EPOCHS
            cls.ENCODER_FREEZE_EPOCHS = cls.SMOKE_TEST_FREEZE_EPOCHS
            cls.T_MAX = cls.SMOKE_TEST_EPOCHS
            cls.BATCH_SIZE = cls.SMOKE_TEST_BATCH_SIZE
            cls.GRADIENT_ACCUMULATION_STEPS = cls.SMOKE_TEST_GRAD_ACCUM
            print(f"🔥 SMOKE TEST MODE ENABLED: {cls.SMOKE_TEST_SAMPLES} samples, {cls.SMOKE_TEST_EPOCHS} epochs")
    
    @classmethod
    def print_config(cls):
        print("=" * 60)
        print("DUAL-ENCODER CONFIGURATION")
        print("=" * 60)
        for key, value in cls.__dict__.items():
            if not key.startswith('_') and not callable(value):
                print(f"{key}: {value}")
        print("=" * 60)

Writing config.py


In [2]:
%%writefile utils.py
"""
Utility functions: RLE encoding/decoding and OF1 metric
Save as: utils.py
"""

import json
import numba
import numpy as np
from numba import types
import numpy.typing as npt
import pandas as pd
import scipy.optimize


class ParticipantVisibleError(Exception):
    pass


@numba.jit(nopython=True)
def _rle_encode_jit(x: npt.NDArray, fg_val: int = 1) -> list[int]:
    """Numba-jitted RLE encoder."""
    dots = np.where(x.T.flatten() == fg_val)[0]
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return run_lengths


def rle_encode(masks: list[npt.NDArray], fg_val: int = 1) -> str:
    """
    Encode masks to RLE format.
    Args:
        masks: list of numpy array of shape (height, width), 1 - mask, 0 - background
    Returns: run length encodings as a string, with each RLE JSON-encoded and separated by a semicolon.
    """
    return ';'.join([json.dumps(_rle_encode_jit(x, fg_val)) for x in masks])


@numba.njit
def _rle_decode_jit(mask_rle: npt.NDArray, height: int, width: int) -> npt.NDArray:
    """
    Decode RLE to mask.
    s: numpy array of run-length encoding pairs (start, length)
    shape: (height, width) of array to return
    Returns numpy array, 1 - mask, 0 - background
    """
    if len(mask_rle) % 2 != 0:
        raise ValueError('One or more rows has an odd number of values.')

    starts, lengths = mask_rle[0::2], mask_rle[1::2]
    starts -= 1
    ends = starts + lengths
    for i in range(len(starts) - 1):
        if ends[i] > starts[i + 1]:
            raise ValueError('Pixels must not be overlapping.')
    img = np.zeros(height * width, dtype=np.bool_)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img


def rle_decode(mask_rle: str, shape: tuple[int, int]) -> npt.NDArray:
    """
    Decode RLE string to mask.
    mask_rle: run-length as string formatted (start length)
              empty predictions need to be encoded with '-'
    shape: (height, width) of array to return
    Returns numpy array, 1 - mask, 0 - background
    """
    mask_rle = json.loads(mask_rle)
    mask_rle = np.asarray(mask_rle, dtype=np.int32)
    starts = mask_rle[0::2]
    if sorted(starts) != list(starts):
        raise ParticipantVisibleError('Submitted values must be in ascending order.')
    try:
        return _rle_decode_jit(mask_rle, shape[0], shape[1]).reshape(shape, order='F')
    except ValueError as e:
        raise ParticipantVisibleError(str(e)) from e


def calculate_f1_score(pred_mask: npt.NDArray, gt_mask: npt.NDArray):
    """Calculate F1 score between two binary masks."""
    pred_flat = pred_mask.flatten()
    gt_flat = gt_mask.flatten()

    tp = np.sum((pred_flat == 1) & (gt_flat == 1))
    fp = np.sum((pred_flat == 1) & (gt_flat == 0))
    fn = np.sum((pred_flat == 0) & (gt_flat == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    if (precision + recall) > 0:
        return 2 * (precision * recall) / (precision + recall)
    else:
        return 0


def calculate_f1_matrix(pred_masks: list[npt.NDArray], gt_masks: list[npt.NDArray]):
    """
    Calculate F1 matrix for all pairs of predicted and ground truth masks.
    """
    num_instances_pred = len(pred_masks)
    num_instances_gt = len(gt_masks)
    f1_matrix = np.zeros((num_instances_pred, num_instances_gt))

    for i in range(num_instances_pred):
        for j in range(num_instances_gt):
            pred_flat = pred_masks[i].flatten()
            gt_flat = gt_masks[j].flatten()
            f1_matrix[i, j] = calculate_f1_score(pred_mask=pred_flat, gt_mask=gt_flat)

    if f1_matrix.shape[0] < len(gt_masks):
        # Add rows of zeros if fewer predictions than ground truth
        f1_matrix = np.vstack((f1_matrix, np.zeros((len(gt_masks) - len(f1_matrix), num_instances_gt))))

    return f1_matrix


def oF1_score(pred_masks: list[npt.NDArray], gt_masks: list[npt.NDArray]):
    """
    Calculate optimal F1 score using Hungarian algorithm.
    """
    f1_matrix = calculate_f1_matrix(pred_masks, gt_masks)
    row_ind, col_ind = scipy.optimize.linear_sum_assignment(-f1_matrix)
    excess_predictions_penalty = len(gt_masks) / max(len(pred_masks), len(gt_masks))
    return np.mean(f1_matrix[row_ind, col_ind]) * excess_predictions_penalty


def evaluate_single_image(label_rles: str, prediction_rles: str, shape_str: str) -> float:
    """Evaluate single image with RLE inputs."""
    shape = json.loads(shape_str)
    label_rles = [rle_decode(x, shape=shape) for x in label_rles.split(';')]
    prediction_rles = [rle_decode(x, shape=shape) for x in prediction_rles.split(';')]
    return oF1_score(prediction_rles, label_rles)


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculate overall score for entire dataset.
    """
    df = solution
    df = df.rename(columns={'annotation': 'label'})

    df['prediction'] = submission['annotation']
    # Check for correct 'authentic' label
    authentic_indices = (df['label'] == 'authentic') | (df['prediction'] == 'authentic')
    df['image_score'] = ((df['label'] == df['prediction']) & authentic_indices).astype(float)

    df.loc[~authentic_indices, 'image_score'] = df.loc[~authentic_indices].apply(
        lambda row: evaluate_single_image(row['label'], row['prediction'], row['shape']), axis=1
    )
    return float(np.mean(df['image_score']))

Writing utils.py


In [3]:
%%writefile dataset.py
"""
Dataset and DataLoader implementation for Dual-Encoder Model
Save as: dataset.py
"""

import os
import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from PIL import Image


class ForgeryDataset(Dataset):
    def __init__(self, authentic_dir, forged_dir, mask_dir, transform=None, config=None):
        self.authentic_dir = authentic_dir
        self.forged_dir = forged_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.config = config
        self.warning_count = 0
        self.max_warnings = 10
        
        self.authentic_images = []
        self.forged_images = []
        
        if os.path.exists(authentic_dir) and os.path.isdir(authentic_dir):
            self.authentic_images = sorted([f for f in os.listdir(authentic_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        
        if os.path.exists(forged_dir) and os.path.isdir(forged_dir):
            self.forged_images = sorted([f for f in os.listdir(forged_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        
        self.samples = []
        for img in self.authentic_images:
            self.samples.append({'image': img, 'is_forged': False, 'dir': authentic_dir})
        for img in self.forged_images:
            self.samples.append({'image': img, 'is_forged': True, 'dir': forged_dir})
        
        print(f"📦 Total samples before limiting: {len(self.samples)} ({len(self.authentic_images)} authentic, {len(self.forged_images)} forged)")
        
        if config and config.SMOKE_TEST:
            original_len = len(self.samples)
            authentic_samples = [s for s in self.samples if not s['is_forged']]
            forged_samples = [s for s in self.samples if s['is_forged']]
            
            random.shuffle(authentic_samples)
            random.shuffle(forged_samples)

            n_samples = config.SMOKE_TEST_SAMPLES
            n_authentic = min(len(authentic_samples), n_samples // 2)
            n_forged = min(len(forged_samples), n_samples - n_authentic)
            if n_forged < n_samples // 2:
                n_authentic = min(len(authentic_samples), n_samples - n_forged)
            
            selected_authentic = authentic_samples[:n_authentic]
            selected_forged = forged_samples[:n_forged]
            self.samples = selected_authentic + selected_forged
            print(f"🔥 SMOKE TEST: Randomly sampled {len(self.samples)} images from {original_len}")
    
    def __len__(self):
        return len(self.samples)
    
    def _verify_and_normalize(self, tensor, name):
        min_val = tensor.min().item()
        max_val = tensor.max().item()
        if min_val < 0.0 or max_val > 1.0:
            if self.warning_count < self.max_warnings:
                print(f"⚠️ SAFETY CHECK WARNING: {name} out of range [{min_val:.3f}, {max_val:.3f}]. Normalizing...")
                self.warning_count += 1
            if max_val > 1.0:
                tensor = tensor / 255.0
            tensor = torch.clamp(tensor, 0.0, 1.0)
        return tensor

    def letterbox_image(self, image, masks, target_size):
        """Letterbox image and masks to target size"""
        h, w = image.shape[:2]
        scale = min(target_size / h, target_size / w)
        new_h, new_w = int(h * scale), int(w * scale)
        
        resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        
        resized_masks = []
        for mask in masks:
            resized_mask = cv2.resize(mask.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST)
            resized_masks.append(resized_mask)
        
        padded_image = np.zeros((target_size, target_size, 3), dtype=np.uint8)
        y_offset = (target_size - new_h) // 2
        x_offset = (target_size - new_w) // 2
        padded_image[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized_image
        
        padded_masks = []
        for mask in resized_masks:
            padded_mask = np.zeros((target_size, target_size), dtype=np.uint8)
            padded_mask[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = mask
            padded_masks.append(padded_mask)
        
        return padded_image, padded_masks
    
    def __getitem__(self, idx):
        try:
            sample = self.samples[idx]
            img_name = sample['image']
            is_forged = sample['is_forged']
            img_dir = sample['dir']
            
            img_path = os.path.join(img_dir, img_name)
            image = cv2.imread(img_path)
            
            if image is None:
                print(f"⚠️  CORRUPT IMAGE DETECTED: {img_name}. Skipping and replacing with random sample.")
                new_idx = random.randint(0, len(self.samples) - 1)
                return self.__getitem__(new_idx)
            
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            original_h, original_w = image.shape[:2]
            
            # Load masks
            masks = []
            if is_forged:
                mask_name = os.path.splitext(img_name)[0] + '.npy'
                mask_path = os.path.join(self.mask_dir, mask_name)
                
                if os.path.exists(mask_path):
                    mask_data = np.load(mask_path, allow_pickle=True)
                    if mask_data.ndim == 2 and np.sum(mask_data) > 0:
                        if mask_data.shape != (original_h, original_w):
                            mask_data = cv2.resize(mask_data.astype(np.uint8), (original_w, original_h), interpolation=cv2.INTER_NEAREST)
                        masks.append(mask_data.astype(np.float32))
                    elif mask_data.ndim == 3:
                        for i in range(mask_data.shape[0]):
                            instance_mask = mask_data[i]
                            if np.sum(instance_mask) > 0:
                                if instance_mask.shape != (original_h, original_w):
                                    instance_mask = cv2.resize(instance_mask.astype(np.uint8), (original_w, original_h), interpolation=cv2.INTER_NEAREST)
                                masks.append(instance_mask.astype(np.float32))
            
            # Create dual inputs: 224×224 and 448×448
            image_224, masks_224 = self.letterbox_image(image, masks, target_size=self.config.INPUT_SIZE_SWIN)
            image_448, masks_448 = self.letterbox_image(image, masks, target_size=self.config.INPUT_SIZE_CONVNEXT)
            
            # Apply augmentations if provided
            if self.transform:
                if len(masks_224) > 0:
                    transformed_224 = self.transform(image=image_224, masks=masks_224)
                    image_224 = transformed_224['image']
                    masks_224 = transformed_224['masks']
                    
                    transformed_448 = self.transform(image=image_448, masks=masks_448)
                    image_448 = transformed_448['image']
                    masks_448 = transformed_448['masks']
                else:
                    transformed_224 = self.transform(image=image_224)
                    image_224 = transformed_224['image']
                    
                    transformed_448 = self.transform(image=image_448)
                    image_448 = transformed_448['image']
            
            # Pad masks to NUM_INSTANCES
            h_224, w_224 = image_224.shape[:2]
            while len(masks_224) < self.config.NUM_INSTANCES:
                masks_224.append(np.zeros((h_224, w_224), dtype=np.float32))
            masks_224 = masks_224[:self.config.NUM_INSTANCES]
            
            # Convert to tensors
            image_224 = torch.from_numpy(image_224).permute(2, 0, 1).float() / 255.0
            image_448 = torch.from_numpy(image_448).permute(2, 0, 1).float() / 255.0
            
            masks_float = []
            for m in masks_224:
                m = m.astype(np.float32)
                masks_float.append(torch.from_numpy(m))
            
            masks = torch.stack(masks_float)
            
            image_224 = self._verify_and_normalize(image_224, f"Image 224 {img_name}")
            image_448 = self._verify_and_normalize(image_448, f"Image 448 {img_name}")
            masks = self._verify_and_normalize(masks, f"Masks {img_name}")
            
            return {
                'input_224': image_224,
                'input_448': image_448,
                'masks': masks,
                'is_forged': torch.tensor(is_forged, dtype=torch.float32),
                'num_instances': torch.tensor(len([m for m in masks if m.sum() > 0]), dtype=torch.long),
                'image_name': img_name
            }
        
        except Exception as e:
            print(f"⚠️  ERROR processing {self.samples[idx]['image']}: {str(e)}. Skipping.")
            new_idx = random.randint(0, len(self.samples) - 1)
            return self.__getitem__(new_idx)


def get_train_transform(config):
    transforms_list = []
    
    if config.AUG_PROB_FLIP > 0:
        transforms_list.extend([
            A.HorizontalFlip(p=config.AUG_PROB_FLIP),
            A.VerticalFlip(p=config.AUG_PROB_FLIP)
        ])
    
    if config.AUG_PROB_ROTATE > 0:
        transforms_list.append(
            A.Rotate(
                limit=config.AUG_ROTATION_LIMIT, 
                border_mode=cv2.BORDER_CONSTANT, 
                p=config.AUG_PROB_ROTATE
            )
        )
        
    if config.AUG_PROB_BLUR > 0:
        transforms_list.append(
            A.GaussianBlur(blur_limit=config.AUG_BLUR_LIMIT, p=config.AUG_PROB_BLUR)
        )
    
    if config.AUG_PROB_NOISE > 0:
        transforms_list.append(
            A.GaussNoise(var_limit=config.AUG_NOISE_VAR_LIMIT, p=config.AUG_PROB_NOISE)
        )
        
    if config.AUG_PROB_JPEG > 0:
        transforms_list.append(
            A.ImageCompression(
                quality_range=config.AUG_JPEG_QUALITY_RANGE, 
                p=config.AUG_PROB_JPEG
            )
        )

    if config.AUG_PROB_COLOR > 0:
        transforms_list.append(
            A.ColorJitter(
                brightness=config.AUG_BRIGHTNESS,
                contrast=config.AUG_CONTRAST,
                saturation=config.AUG_SATURATION,
                hue=config.AUG_HUE,
                p=config.AUG_PROB_COLOR
            )
        )

    return A.Compose(transforms_list)


def get_val_transform(config):
    return None


def create_dataloaders(config, rank=0, world_size=1):
    train_dataset = ForgeryDataset(
        authentic_dir=config.TRAIN_AUTHENTIC_DIR,
        forged_dir=config.TRAIN_FORGED_DIR,
        mask_dir=config.TRAIN_MASK_DIR,
        transform=get_train_transform(config),
        config=config
    )
    
    val_dataset = ForgeryDataset(
        authentic_dir=config.TEST_AUTHENTIC_DIR,
        forged_dir=config.TEST_FORGED_DIR,
        mask_dir=config.TEST_MASK_DIR,
        transform=get_val_transform(config),
        config=config
    )
    
    train_sampler = DistributedSampler(
        train_dataset, num_replicas=world_size, rank=rank, shuffle=True, drop_last=True
    )
    
    val_sampler = DistributedSampler(
        val_dataset, num_replicas=world_size, rank=rank, shuffle=False, drop_last=False
    )
    
    train_loader = DataLoader(
        train_dataset, batch_size=config.BATCH_SIZE, sampler=train_sampler, 
        num_workers=4, pin_memory=True, drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset, batch_size=config.BATCH_SIZE, sampler=val_sampler, 
        num_workers=4, pin_memory=True, drop_last=False
    )
    
    return train_loader, val_loader, train_sampler, val_sampler

Writing dataset.py


In [4]:
%%writefile model.py
"""
Dual-Encoder Model: Swin-Small + ConvNeXt-Small with Auxiliary Heads
Save as: model.py
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm


class CoordConv(nn.Module):
    """Add normalized x,y coordinate channels"""
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        B, C, H, W = x.shape
        device = x.device
        
        # Create normalized coordinate grids
        y_coords = torch.linspace(-1, 1, H, device=device).view(1, 1, H, 1).repeat(B, 1, 1, W)
        x_coords = torch.linspace(-1, 1, W, device=device).view(1, 1, 1, W).repeat(B, 1, H, 1)
        
        # Concatenate coordinates as additional channels
        x_with_coords = torch.cat([x, x_coords, y_coords], dim=1)
        return x_with_coords


class DoubleConv(nn.Module):
    """Double convolution block for U-Net decoder"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)


class ConvNeXtDownsample(nn.Module):
    """Downsample ConvNeXt features to match Swin resolution"""
    def __init__(self, channels):
        super().__init__()
        self.downsample = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        return self.relu(self.bn(self.downsample(x)))


class FusionBlock(nn.Module):
    """Fusion block: Normalize -> Concat -> Channel reduction to 2/3"""
    def __init__(self, in_channels, reduction_ratio=2/3):
        super().__init__()
        # in_channels is per encoder, total will be 2*in_channels
        total_channels = 2 * in_channels
        out_channels = int(total_channels * reduction_ratio)
        
        self.bn_swin = nn.BatchNorm2d(in_channels)
        self.bn_convnext = nn.BatchNorm2d(in_channels)
        self.fusion_conv = nn.Sequential(
            nn.Conv2d(total_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, swin_feat, convnext_feat):
        swin_feat = self.bn_swin(swin_feat)
        convnext_feat = self.bn_convnext(convnext_feat)
        fused = torch.cat([swin_feat, convnext_feat], dim=1)
        return self.fusion_conv(fused)


class Up(nn.Module):
    """Upsampling block with optional skip connection"""
    def __init__(self, in_channels, out_channels, skip_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels // 2 + skip_channels, out_channels)
        self.has_skip = skip_channels > 0
    
    def forward(self, x, skip=None):
        x = self.up(x)
        if self.has_skip and skip is not None:
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class SwinContrastiveHead(nn.Module):
    """Aux Head 1: Mini U-Net from Swin bottleneck (7×7) to 56×56 embeddings"""
    def __init__(self, in_channels, embedding_dim):
        super().__init__()
        # Mini decoder: 7×7 → 14×14 → 28×28 → 56×56
        self.up1 = nn.ConvTranspose2d(in_channels, 384, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(384, 384)
        
        self.up2 = nn.ConvTranspose2d(384, 192, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(192, 192)
        
        self.up3 = nn.ConvTranspose2d(192, 96, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(96, 96)
        
        # Embedding projection
        self.embedding_proj = nn.Conv2d(96, embedding_dim, kernel_size=1)
    
    def forward(self, x):
        # x: (B, 768, 7, 7)
        x = self.up1(x)  # (B, 384, 14, 14)
        x = self.conv1(x)
        
        x = self.up2(x)  # (B, 192, 28, 28)
        x = self.conv2(x)
        
        x = self.up3(x)  # (B, 96, 56, 56)
        x = self.conv3(x)
        
        embeddings = self.embedding_proj(x)  # (B, 128, 56, 56)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings


class ConvNeXtCoarseHead(nn.Module):
    """Aux Head 2: ConvNeXt bottleneck (14×14) to coarse mask (28×28)"""
    def __init__(self, in_channels):
        super().__init__()
        # Gradient-safe block
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv2 = nn.Sequential(
            nn.Conv2d(256, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.final = nn.Conv2d(64, 1, kernel_size=1)  # Output logits
    
    def forward(self, x):
        # x: (B, 768, 14, 14)
        x = self.conv1(x)  # (B, 256, 14, 14)
        x = self.upsample(x)  # (B, 256, 28, 28)
        x = self.conv2(x)  # (B, 64, 28, 28)
        logits = self.final(x)  # (B, 1, 28, 28)
        return logits


class ActiveChannelHead(nn.Module):
    """Active channel prediction with convolutional neck"""
    def __init__(self, in_channels, num_channels=6):
        super().__init__()
        # Convolutional neck: 224×224 → 28×28
        self.neck = nn.Sequential(
            # 224 → 112
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            
            # 112 → 56
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            # 56 → 28
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        
        # Global pooling (max + avg)
        self.global_max_pool = nn.AdaptiveMaxPool2d(1)
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        
        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(512, 256),  # 256 from max + 256 from avg
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_channels)
        )
    
    def forward(self, x):
        # x: (B, 32, 224, 224)
        x = self.neck(x)  # (B, 256, 28, 28)
        
        max_pool = self.global_max_pool(x).view(x.size(0), -1)  # (B, 256)
        avg_pool = self.global_avg_pool(x).view(x.size(0), -1)  # (B, 256)
        
        pooled = torch.cat([max_pool, avg_pool], dim=1)  # (B, 512)
        logits = self.fc(pooled)  # (B, 6)
        return logits


class ForgeryDetectionModel(nn.Module):
    """
    Dual-Encoder Copy-Move Forgery Detection Model
    - Swin-Small encoder (224×224) - Learns similarity
    - ConvNeXt-Small encoder (448×448) - Learns coarse forgery detection
    - Fused decoder with skip connections
    - CoordConv at 1/4 stage (up3 layer)
    - Three heads: Main Segmentation, Active Channel, Aux Contrastive (Swin), Aux Coarse (ConvNeXt)
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # ==================== DUAL ENCODERS ====================
        self.swin_encoder = timm.create_model(
            config.SWIN_MODEL,
            pretrained=True,
            features_only=True,
            out_indices=[0, 1, 2, 3]
        )
        
        self.convnext_encoder = timm.create_model(
            config.CONVNEXT_MODEL,
            pretrained=True,
            features_only=True,
            out_indices=[0, 1, 2, 3]
        )
        
        # Feature dimensions
        # Swin: [96@56×56, 192@28×28, 384@14×14, 768@7×7]
        # ConvNeXt: [96@112×112, 192@56×56, 384@28×28, 768@14×14]
        swin_dims = [96, 192, 384, 768]
        convnext_dims = [96, 192, 384, 768]
        
        # ==================== DOWNSAMPLING BLOCKS ====================
        self.convnext_down0 = ConvNeXtDownsample(convnext_dims[0])  # 112 → 56
        self.convnext_down1 = ConvNeXtDownsample(convnext_dims[1])  # 56 → 28
        self.convnext_down2 = ConvNeXtDownsample(convnext_dims[2])  # 28 → 14
        self.convnext_down3 = ConvNeXtDownsample(convnext_dims[3])  # 14 → 7
        
        # ==================== FUSION BLOCKS ====================
        self.fusion0 = FusionBlock(swin_dims[0], config.FUSION_REDUCTION_RATIO)
        self.fusion1 = FusionBlock(swin_dims[1], config.FUSION_REDUCTION_RATIO)
        self.fusion2 = FusionBlock(swin_dims[2], config.FUSION_REDUCTION_RATIO)
        self.fusion3 = FusionBlock(swin_dims[3], config.FUSION_REDUCTION_RATIO)
        
        # Fused channel dimensions (2/3 of 2*C)
        fused_dims = [int(2 * c * config.FUSION_REDUCTION_RATIO) for c in swin_dims]
        
        # ==================== DECODER ====================
        decoder_dims = config.DECODER_CHANNELS
        
        self.bottleneck = DoubleConv(fused_dims[3], decoder_dims[0])
        self.up1 = Up(decoder_dims[0], decoder_dims[1], fused_dims[2])
        self.up2 = Up(decoder_dims[1], decoder_dims[2], fused_dims[1])
        
        # ==================== COORDCONV AT 1/4 STAGE ====================
        self.coord_conv = CoordConv()
        # up3 receives decoder_dims[2] + 2 (coords) as input instead of just decoder_dims[2]
        self.up3 = Up(decoder_dims[2] + 2, decoder_dims[3], fused_dims[0])
        
        self.up4 = Up(decoder_dims[3], decoder_dims[4], 0)
        
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(decoder_dims[4], decoder_dims[4], kernel_size=2, stride=2),
            nn.BatchNorm2d(decoder_dims[4]),
            nn.ReLU(inplace=True)
        )
        
        # ==================== AUXILIARY HEADS ====================
        # Aux Head 1: Swin Contrastive (7×7 → 56×56 embeddings)
        self.swin_contrastive_head = SwinContrastiveHead(
            in_channels=swin_dims[3],
            embedding_dim=config.EMBEDDING_DIM
        )
        
        # Aux Head 2: ConvNeXt Coarse Mask (14×14 → 28×28)
        self.convnext_coarse_head = ConvNeXtCoarseHead(
            in_channels=convnext_dims[3]
        )
        
        # ==================== MAIN HEADS ====================
        # Active Channel Head
        self.active_channel_head = ActiveChannelHead(
            in_channels=decoder_dims[4],
            num_channels=config.NUM_INSTANCES
        )
        
        # Segmentation Head with smoothing
        self.segmentation_head = nn.Sequential(
            nn.Conv2d(decoder_dims[4], decoder_dims[4], kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(decoder_dims[4]),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dims[4], config.NUM_INSTANCES, kernel_size=1)
        )
        
        self.freeze_encoders()
    
    def freeze_encoders(self):
        """Freeze both encoders"""
        for param in self.swin_encoder.parameters():
            param.requires_grad = False
        for param in self.convnext_encoder.parameters():
            param.requires_grad = False
        print("🧊 Both encoders frozen")
    
    def unfreeze_encoders(self):
        """Unfreeze both encoders"""
        for param in self.swin_encoder.parameters():
            param.requires_grad = True
        for param in self.convnext_encoder.parameters():
            param.requires_grad = True
        print("🔥 Both encoders unfrozen")
    
    def process_swin_features(self, features):
        """Convert Swin features to NCHW if needed"""
        processed = []
        for feat in features:
            if feat.dim() == 4 and feat.shape[1] < feat.shape[3]:
                feat = feat.permute(0, 3, 1, 2)
            processed.append(feat)
        return processed
    
    def forward(self, x_swin, x_convnext):
        """
        Forward pass with dual inputs
        Args:
            x_swin: (B, 3, 224, 224)
            x_convnext: (B, 3, 448, 448)
        Returns:
            Dictionary with outputs from all heads
        """
        # ==================== ENCODE ====================
        swin_features = self.swin_encoder(x_swin)
        swin_features = self.process_swin_features(swin_features)
        
        convnext_features = self.convnext_encoder(x_convnext)
        
        # Swin: [96@56×56, 192@28×28, 384@14×14, 768@7×7]
        # ConvNeXt: [96@112×112, 192@56×56, 384@28×28, 768@14×14]
        
        # ==================== AUXILIARY HEADS (Pre-Fusion) ====================
        # Aux Head 1: Swin Contrastive
        swin_embeddings = self.swin_contrastive_head(swin_features[3])  # (B, 128, 56, 56)
        
        # Aux Head 2: ConvNeXt Coarse Mask
        convnext_coarse_logits = self.convnext_coarse_head(convnext_features[3])  # (B, 1, 28, 28)
        
        # ==================== DOWNSAMPLE CONVNEXT ====================
        convnext_down0 = self.convnext_down0(convnext_features[0])  # 112 → 56
        convnext_down1 = self.convnext_down1(convnext_features[1])  # 56 → 28
        convnext_down2 = self.convnext_down2(convnext_features[2])  # 28 → 14
        convnext_down3 = self.convnext_down3(convnext_features[3])  # 14 → 7
        
        # ==================== FUSION ====================
        fused0 = self.fusion0(swin_features[0], convnext_down0)
        fused1 = self.fusion1(swin_features[1], convnext_down1)
        fused2 = self.fusion2(swin_features[2], convnext_down2)
        fused3 = self.fusion3(swin_features[3], convnext_down3)
        
        # ==================== DECODE ====================
        x = self.bottleneck(fused3)
        x = self.up1(x, fused2)
        x = self.up2(x, fused1)
        
        # Add CoordConv at 1/4 stage (after up2, before up3)
        x = self.coord_conv(x)  # Add x,y coords as 2 extra channels
        
        x = self.up3(x, fused0)
        x = self.up4(x, None)
        x = self.final_up(x)  # (B, 32, 224, 224)
        
        # ==================== MAIN HEADS ====================
        active_logits = self.active_channel_head(x)  # (B, 6)
        segmentation_logits = self.segmentation_head(x)  # (B, 6, 224, 224)
        segmentation = torch.sigmoid(segmentation_logits)
        
        return {
            'segmentation': segmentation,
            'segmentation_logits': segmentation_logits,  # For AMP-safe focal loss
            'active_logits': active_logits,
            'swin_embeddings': swin_embeddings,
            'convnext_coarse_logits': convnext_coarse_logits
        }


def create_model(config):
    """Factory function to create model"""
    model = ForgeryDetectionModel(config)
    return model

Writing model.py


In [5]:
%%writefile losses.py
"""
Loss functions for Dual-Encoder Model
Save as: losses.py
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
import numpy as np


class DiceLoss(nn.Module):
    """Dice Loss for segmentation"""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = pred.contiguous().view(pred.size(0), -1)
        target = target.contiguous().view(target.size(0), -1)
        intersection = (pred * target).sum(dim=1)
        dice = (2. * intersection + self.smooth) / (pred.sum(dim=1) + target.sum(dim=1) + self.smooth)
        return 1 - dice.mean()


class SegmentationFocalLoss(nn.Module):
    """Focal Loss for pixel-level segmentation - AMP Safe"""
    def __init__(self, alpha=0.99, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, pred_logits, target):
        """
        Args:
            pred_logits: Raw logits (before sigmoid)
            target: Binary target
        """
        pred_logits = pred_logits.contiguous().view(-1)
        target = target.contiguous().view(-1)
        
        # Use BCE with logits (AMP safe)
        bce = F.binary_cross_entropy_with_logits(pred_logits, target, reduction='none')
        
        # Calculate probabilities for focal weight
        pred_probs = torch.sigmoid(pred_logits)
        pt = torch.where(target == 1, pred_probs, 1 - pred_probs)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = torch.where(target == 1, self.alpha, 1 - self.alpha)
        
        loss = alpha_weight * focal_weight * bce
        return loss.mean()


class ContrastiveLoss(nn.Module):
    """Instance-Aware Pixel Contrastive Loss (for Swin embeddings) - Updated Sampling"""
    def __init__(self, config):
        super().__init__()
        self.temperature = config.CONTRASTIVE_TEMPERATURE
        self.max_samples_bg = config.CONTRASTIVE_SAMPLES_BACKGROUND
        self.max_samples_inst = config.CONTRASTIVE_SAMPLES_PER_INSTANCE
        self.margin_bg_fg = config.CONTRASTIVE_MARGIN_BG_FG
        self.margin_fg_fg = config.CONTRASTIVE_MARGIN_FG_FG
        self.w_bg_fg = config.CONTRAST_WEIGHT_BG_FG
        self.w_fg_fg = config.CONTRAST_WEIGHT_FG_FG
    
    def forward(self, embeddings, masks):
        """
        Args:
            embeddings: (B, D, 56, 56) - Normalized
            masks: (B, 6, 224, 224)
        """
        B, D, H, W = embeddings.shape
        device = embeddings.device
        
        # Downsample masks to embedding resolution
        masks_down = F.interpolate(masks, size=(H, W), mode='nearest')
        
        total_loss = 0.0
        valid_batches = 0
        
        for b in range(B):
            emb = embeddings[b].view(D, -1).t()  # (Pixels, D)
            curr_masks = masks_down[b]
            
            # Create label map
            label_map = torch.zeros((H, W), device=device, dtype=torch.long)
            active_instance_ids = []
            
            for i in range(curr_masks.shape[0]):
                mask_i = curr_masks[i]
                if mask_i.sum() > 0:
                    label_map[mask_i > 0.5] = (i + 1)
                    active_instance_ids.append(i + 1)
            
            label_flat = label_map.flatten()
            
            # Stratified sampling with different limits for BG vs instances
            sampled_embeddings = []
            sampled_labels = []
            
            # Sample background (more samples)
            bg_indices = torch.where(label_flat == 0)[0]
            if len(bg_indices) > 0:
                n_bg = min(len(bg_indices), self.max_samples_bg)
                perm = torch.randperm(len(bg_indices), device=device)[:n_bg]
                chosen_bg = bg_indices[perm]
                sampled_embeddings.append(emb[chosen_bg])
                sampled_labels.append(label_flat[chosen_bg])
            
            # Sample each instance (fewer samples per instance)
            for inst_id in active_instance_ids:
                inst_indices = torch.where(label_flat == inst_id)[0]
                if len(inst_indices) > 0:
                    n_inst = min(len(inst_indices), self.max_samples_inst)
                    perm = torch.randperm(len(inst_indices), device=device)[:n_inst]
                    chosen_inst = inst_indices[perm]
                    sampled_embeddings.append(emb[chosen_inst])
                    sampled_labels.append(label_flat[chosen_inst])
            
            if len(sampled_labels) == 0:
                continue
            
            anchors = torch.cat(sampled_embeddings, dim=0)
            anchor_labels = torch.cat(sampled_labels, dim=0)
            
            num_anchors = anchors.shape[0]
            if num_anchors < 2:
                continue
            
            # Distance matrix
            dist_matrix = torch.cdist(anchors, anchors, p=2)
            
            # Masks
            mask_same_class = anchor_labels.unsqueeze(0) == anchor_labels.unsqueeze(1)
            is_bg = anchor_labels == 0
            mask_bg_fg = is_bg.unsqueeze(0) ^ is_bg.unsqueeze(1)
            mask_fg_fg = (~mask_same_class) & (~mask_bg_fg)
            
            # Pull loss
            mask_same_class.fill_diagonal_(False)
            if mask_same_class.sum() > 0:
                pos_loss = dist_matrix[mask_same_class].mean()
            else:
                pos_loss = torch.tensor(0.0, device=device)
            
            # Push losses
            if mask_bg_fg.sum() > 0:
                dists = dist_matrix[mask_bg_fg]
                neg_loss_bg_fg = F.relu(self.margin_bg_fg - dists).mean()
            else:
                neg_loss_bg_fg = torch.tensor(0.0, device=device)
            
            if mask_fg_fg.sum() > 0:
                dists = dist_matrix[mask_fg_fg]
                neg_loss_fg_fg = F.relu(self.margin_fg_fg - dists).mean()
            else:
                neg_loss_fg_fg = torch.tensor(0.0, device=device)
            
            batch_loss = pos_loss + (self.w_bg_fg * neg_loss_bg_fg) + (self.w_fg_fg * neg_loss_fg_fg)
            total_loss += batch_loss
            valid_batches += 1
        
        if valid_batches > 0:
            return total_loss / valid_batches
        else:
            return 0.0 * embeddings.sum()


class CoarseMaskLoss(nn.Module):
    """BCE Loss for ConvNeXt coarse forgery mask"""
    def __init__(self):
        super().__init__()
        self.bce_loss = nn.BCEWithLogitsLoss()
    
    def forward(self, coarse_logits, gt_masks):
        """
        Args:
            coarse_logits: (B, 1, 28, 28)
            gt_masks: (B, 6, 224, 224)
        Returns:
            BCE loss
        """
        # Merge all instance masks into single channel (logical OR)
        merged_masks = (gt_masks.sum(dim=1, keepdim=True) > 0).float()  # (B, 1, 224, 224)
        
        # Downsample to 28×28 using max pooling
        coarse_gt = F.max_pool2d(merged_masks, kernel_size=8, stride=8)  # (B, 1, 28, 28)
        
        loss = self.bce_loss(coarse_logits, coarse_gt)
        return loss


class FocalLoss(nn.Module):
    """Focal Loss for active channel prediction"""
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        loss = alpha_weight * focal_weight * bce_loss
        return loss.mean()


def hungarian_matching_fixed(pred_masks, gt_masks):
    """Hungarian matching for segmentation"""
    B = pred_masks.shape[0]
    indices = []
    
    for b in range(B):
        pred = pred_masks[b]
        gt = gt_masks[b]
        
        active_gt_indices = []
        for i in range(6):
            if gt[i].sum() > 0:
                active_gt_indices.append(i)
        
        num_active_gt = len(active_gt_indices)
        cost_matrix = np.zeros((6, 6))
        
        for i in range(6):
            pred_flat = pred[i].flatten().detach().cpu().numpy()
            pred_sum = pred_flat.sum()
            
            for j in range(num_active_gt):
                gt_idx = active_gt_indices[j]
                gt_flat = gt[gt_idx].flatten().detach().cpu().numpy()
                
                intersection = (pred_flat * gt_flat).sum()
                union = pred_sum + gt_flat.sum()
                dice = (2.0 * intersection) / (union + 1e-6) if union > 0 else 0.0
                cost_matrix[i, j] = 1.0 - dice
            
            for j in range(num_active_gt, 6):
                total_pixels = pred_flat.shape[0]
                cost_matrix[i, j] = pred_sum / (total_pixels + 1e-6)
        
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        
        mapping = {}
        for r, c in zip(row_ind, col_ind):
            if c < num_active_gt:
                mapping[r] = active_gt_indices[c]
            else:
                mapping[r] = -1
        
        indices.append(mapping)
    
    return indices


class CombinedLoss(nn.Module):
    """
    Combined Loss with Dual-Encoder Auxiliary Heads
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Main decoder losses
        self.dice_loss = DiceLoss()
        self.seg_focal_loss = SegmentationFocalLoss(
            alpha=config.SEG_FOCAL_ALPHA,
            gamma=config.SEG_FOCAL_GAMMA
        )
        self.focal_loss = FocalLoss(
            alpha=config.FOCAL_ALPHA,
            gamma=config.FOCAL_GAMMA
        )
        
        # Auxiliary losses
        self.contrastive_loss = ContrastiveLoss(config)
        self.coarse_mask_loss = CoarseMaskLoss()
    
    def forward(self, outputs, targets):
        pred_masks = outputs['segmentation']
        segmentation_logits = outputs.get('segmentation_logits', None)
        gt_masks = targets['masks']
        active_logits = outputs['active_logits']
        swin_embeddings = outputs['swin_embeddings']
        convnext_coarse_logits = outputs['convnext_coarse_logits']
        
        device = pred_masks.device
        B = pred_masks.shape[0]
        
        # ==================== MAIN DECODER LOSSES ====================
        with torch.no_grad():
            mappings = hungarian_matching_fixed(pred_masks, gt_masks)
        
        dice_loss_accum = 0.0
        seg_focal_loss_accum = 0.0
        num_active_matches = 0
        active_targets = torch.zeros_like(active_logits)
        
        for b in range(B):
            mapping = mappings[b]
            for pred_idx in range(6):
                gt_idx = mapping[pred_idx]
                if gt_idx != -1:
                    active_targets[b, pred_idx] = 1.0
                    target_mask = gt_masks[b, gt_idx]
                    
                    dice_loss_accum += self.dice_loss(
                        pred_masks[b, pred_idx].unsqueeze(0),
                        target_mask.unsqueeze(0)
                    )
                    seg_focal_loss_accum += self.seg_focal_loss(
                        segmentation_logits[b, pred_idx],
                        target_mask
                    )
                    num_active_matches += 1
        
        if num_active_matches > 0:
            dice_loss_val = dice_loss_accum / num_active_matches
            seg_focal_loss_val = seg_focal_loss_accum / num_active_matches
        else:
            dice_loss_val = 0.0 * pred_masks.sum()
            seg_focal_loss_val = 0.0 * pred_masks.sum()
        
        focal_loss_val = self.focal_loss(active_logits, active_targets)
        
        # Apply main decoder scaling
        main_decoder_loss = self.config.MAIN_DECODER_SCALE * (
            self.config.LOSS_WEIGHT_DICE * dice_loss_val +
            self.config.LOSS_WEIGHT_SEGMENTATION_FOCAL * seg_focal_loss_val +
            self.config.LOSS_WEIGHT_ACTIVE * focal_loss_val
        )
        
        # ==================== AUXILIARY LOSSES ====================
        # Swin: Contrastive learning on embeddings
        swin_contrastive_loss_val = self.contrastive_loss(swin_embeddings, gt_masks)
        
        # ConvNeXt: Coarse forgery detection
        convnext_coarse_loss_val = self.coarse_mask_loss(convnext_coarse_logits, gt_masks)
        
        # ==================== TOTAL LOSS ====================
        total_loss = (
            main_decoder_loss +
            self.config.LOSS_WEIGHT_SWIN_CONTRASTIVE * swin_contrastive_loss_val +
            self.config.LOSS_WEIGHT_CONVNEXT_COARSE * convnext_coarse_loss_val
        )
        
        loss_dict = {
            'total': total_loss.item(),
            'dice': dice_loss_val.item(),
            'seg_focal': seg_focal_loss_val.item(),
            'focal': focal_loss_val.item(),
            'swin_contrastive': swin_contrastive_loss_val.item(),
            'convnext_coarse': convnext_coarse_loss_val.item()
        }
        
        return total_loss, loss_dict

Writing losses.py


In [6]:
%%writefile train.py
"""
Training script for Dual-Encoder Model - With Early Stopping and Sampled OF1
Save as: train.py
"""
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import numpy as np
from pathlib import Path
import json
import random
from config import Config
from model import create_model
from losses import CombinedLoss
from dataset import create_dataloaders
from utils import oF1_score

def setup_ddp(rank, world_size):
    """Initialize DDP"""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group(Config.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp():
    """Cleanup DDP"""
    dist.destroy_process_group()

def save_checkpoint(model, optimizer, scheduler, scaler, epoch, best_of1, checkpoint_dir, is_best=False):
    """Save model checkpoint"""
    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.module.state_dict() if isinstance(model, DDP) else model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler else None,
        'best_of1': best_of1
    }
    
    current_path = os.path.join(checkpoint_dir, 'current_checkpoint.pth')
    torch.save(checkpoint, current_path)
    if is_best:
        best_path = os.path.join(checkpoint_dir, 'best_checkpoint.pth')
        torch.save(checkpoint, best_path)
        print(f"⭐ Saved BEST checkpoint (OF1: {best_of1:.4f})")

def compute_validation_loss_and_of1(model, dataloader, criterion, config, device, rank):
    """
    Compute validation loss and OF1 score with 50% sampling at 3:7 ratio (authentic:forged)
    """
    model.eval()
    
    # Track validation loss
    val_loss_total = 0.0
    val_loss_dict_sum = {
        'total': 0.0, 
        'dice': 0.0, 
        'seg_focal': 0.0, 
        'focal': 0.0,
        'swin_contrastive': 0.0,
        'convnext_coarse': 0.0
    }
    num_batches = 0
    
    # Collect all samples for OF1
    all_samples = []
    
    with torch.no_grad():
        for batch in dataloader:
            inputs_224 = batch['input_224'].to(device)
            inputs_448 = batch['input_448'].to(device)
            gt_masks = batch['masks'].to(device)
            is_forged = batch['is_forged']
            
            # Calculate validation loss
            targets = {
                'masks': gt_masks,
                'num_instances': batch['num_instances']
            }
            
            outputs = model(inputs_224, inputs_448)
            loss, loss_dict = criterion(outputs, targets)
            
            val_loss_total += loss.item()
            for key in loss_dict:
                val_loss_dict_sum[key] += loss_dict[key]
            num_batches += 1
            
            # Calculate OF1
            pred_masks = outputs['segmentation']
            active_logits = outputs['active_logits']
            
            active_probs = torch.sigmoid(active_logits)
            active_channels = (active_probs > config.ACTIVE_CHANNEL_THRESHOLD).cpu().numpy()
            
            batch_size = inputs_224.shape[0]
            for b in range(batch_size):
                active_indices = np.where(active_channels[b])[0]
                
                pred_masks_list = []
                for idx in active_indices:
                    mask = pred_masks[b, idx].cpu().numpy()
                    mask = (mask > config.MASK_THRESHOLD).astype(np.float32)
                    pred_masks_list.append(mask)
                
                gt_masks_list = []
                for i in range(config.NUM_INSTANCES):
                    mask = gt_masks[b, i].cpu().numpy()
                    if mask.sum() > 0:
                        gt_masks_list.append(mask)
                
                if len(gt_masks_list) == 0:
                    score = 1.0 if len(pred_masks_list) == 0 else 0.0
                else:
                    score = 0.0 if len(pred_masks_list) == 0 else oF1_score(pred_masks_list, gt_masks_list)
                
                all_samples.append({
                    'score': score,
                    'is_forged': is_forged[b].item() > 0.5
                })
    
    model.train()
    
    # Average validation loss
    avg_val_loss = val_loss_total / num_batches if num_batches > 0 else 0.0
    avg_val_loss_dict = {k: v / num_batches for k, v in val_loss_dict_sum.items()} if num_batches > 0 else val_loss_dict_sum
    
    # Separate authentic and forged for OF1
    authentic_samples = [s for s in all_samples if not s['is_forged']]
    forged_samples = [s for s in all_samples if s['is_forged']]
    
    # Sample 50% with 3:7 ratio
    total_samples = len(all_samples)
    target_total = int(total_samples * 0.5)
    target_authentic = int(target_total * 0.3)
    target_forged = target_total - target_authentic
    
    # Random sampling
    sampled_authentic = random.sample(authentic_samples, min(target_authentic, len(authentic_samples)))
    sampled_forged = random.sample(forged_samples, min(target_forged, len(forged_samples)))
    
    # Calculate OF1 scores
    authentic_scores = [s['score'] for s in sampled_authentic]
    forged_scores = [s['score'] for s in sampled_forged]
    
    authentic_of1 = np.mean(authentic_scores) if authentic_scores else 0.0
    forged_of1 = np.mean(forged_scores) if forged_scores else 0.0
    overall_of1 = np.mean(authentic_scores + forged_scores) if (authentic_scores + forged_scores) else 0.0
    
    return avg_val_loss, avg_val_loss_dict, overall_of1, authentic_of1, forged_of1

def train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, epoch, config, device, rank):
    """Train for one epoch with gradient accumulation and mixed precision"""
    model.train()
    
    total_loss = 0.0
    loss_dict_sum = {
        'total': 0.0, 
        'dice': 0.0, 
        'seg_focal': 0.0, 
        'focal': 0.0,
        'swin_contrastive': 0.0,
        'convnext_coarse': 0.0
    }
    
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}", disable=(rank != 0))
    
    for batch_idx, batch in enumerate(pbar):
        inputs_224 = batch['input_224'].to(device)
        inputs_448 = batch['input_448'].to(device)
        masks = batch['masks'].to(device)
        
        targets = {
            'masks': masks,
            'num_instances': batch['num_instances']
        }
        
        # Forward pass with mixed precision
        if config.USE_AMP:
            with autocast():
                outputs = model(inputs_224, inputs_448)
                loss, loss_dict = criterion(outputs, targets)
                loss = loss / config.GRADIENT_ACCUMULATION_STEPS
        else:
            outputs = model(inputs_224, inputs_448)
            loss, loss_dict = criterion(outputs, targets)
            loss = loss / config.GRADIENT_ACCUMULATION_STEPS
        
        # Backward pass
        if config.USE_AMP:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        
        # Optimizer step every GRADIENT_ACCUMULATION_STEPS
        if (batch_idx + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
            if config.USE_AMP:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            
            optimizer.zero_grad()
        
        # Accumulate loss for logging (multiply back by accum steps)
        total_loss += loss.item() * config.GRADIENT_ACCUMULATION_STEPS
        for key in loss_dict:
            loss_dict_sum[key] += loss_dict[key]
        
        if rank == 0:
            pbar.set_postfix({
                'loss': f"{loss.item() * config.GRADIENT_ACCUMULATION_STEPS:.4f}",
                'dice': f"{loss_dict['dice']:.4f}",
                's_cont': f"{loss_dict['swin_contrastive']:.4f}",
                'c_coarse': f"{loss_dict['convnext_coarse']:.4f}"
            })
    
    # Handle remaining gradients if batch count not divisible by accum steps
    if len(train_loader) % config.GRADIENT_ACCUMULATION_STEPS != 0:
        if config.USE_AMP:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad()
    
    num_batches = len(train_loader)
    avg_loss = total_loss / num_batches
    avg_loss_dict = {k: v / num_batches for k, v in loss_dict_sum.items()}
    
    return avg_loss, avg_loss_dict

def train_ddp(rank, world_size, config, smoke_test=False):
    """Main training function with DDP and Early Stopping"""
    setup_ddp(rank, world_size)
    device = rank
    
    if smoke_test:
        config.set_smoke_test(True)
    
    if rank == 0:
        print(f"\n{'='*60}")
        print(f"Starting Dual-Encoder Training on {world_size} GPUs")
        print(f"Batch Size: {config.BATCH_SIZE}, Gradient Accumulation: {config.GRADIENT_ACCUMULATION_STEPS}")
        print(f"Effective Batch Size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS * world_size}")
        print(f"Mixed Precision: {config.USE_AMP}")
        print(f"Early Stopping Patience: {config.EARLY_STOPPING_PATIENCE}")
        print(f"{'='*60}\n")
        config.print_config()
    
    train_loader, val_loader, train_sampler, val_sampler = create_dataloaders(
        config, rank=rank, world_size=world_size
    )
    
    model = create_model(config).to(device)
    model = DDP(model, device_ids=[rank], find_unused_parameters=True)
    
    criterion = CombinedLoss(config).to(device)
    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.T_MAX, eta_min=config.ETA_MIN)
    
    # Initialize GradScaler for AMP
    scaler = GradScaler() if config.USE_AMP else None
    
    best_of1 = -1.0
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    history = {
        'epochs': [],
        'train_loss_total': [],
        'train_loss_dice': [],
        'train_loss_seg_focal': [],
        'train_loss_focal': [],
        'train_loss_swin_contrastive': [],
        'train_loss_convnext_coarse': [],
        'val_loss_total': [],
        'val_loss_dice': [],
        'val_loss_seg_focal': [],
        'val_loss_focal': [],
        'val_loss_swin_contrastive': [],
        'val_loss_convnext_coarse': [],
        'val_of1_overall': [],
        'val_of1_authentic': [],
        'val_of1_forged': [],
        'lr': []
    }
    
    for epoch in range(1, config.NUM_EPOCHS + 1):
        train_sampler.set_epoch(epoch)
        
        if epoch == config.ENCODER_FREEZE_EPOCHS + 1:
            if rank == 0:
                print(f"\n🔥 Unfreezing encoders at epoch {epoch}\n")
            model.module.unfreeze_encoders()
        
        avg_loss, avg_loss_dict = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler, epoch, config, device, rank
        )
        
        scheduler.step()
        
        if rank == 0:
            print(f"\nEpoch {epoch}/{config.NUM_EPOCHS}")
            print(f"Train Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")
            print(f"  Train - DICE: {avg_loss_dict['dice']:.4f}, SegFocal: {avg_loss_dict['seg_focal']:.4f}, "
                  f"Focal: {avg_loss_dict['focal']:.4f}")
            print(f"  Train - Swin Contrastive: {avg_loss_dict['swin_contrastive']:.4f}, "
                  f"ConvNeXt Coarse: {avg_loss_dict['convnext_coarse']:.4f}")
        
        # Validation every VAL_FREQUENCY epochs
        should_validate = (epoch == 1) or (epoch % config.VAL_FREQUENCY == 0) or (epoch == config.NUM_EPOCHS)
        
        if should_validate:
            avg_val_loss, avg_val_loss_dict, overall_of1, authentic_of1, forged_of1 = compute_validation_loss_and_of1(
                model, val_loader, criterion, config, device, rank
            )
            
            if rank == 0:
                print(f"  Val Loss: {avg_val_loss:.4f}")
                print(f"  Val   - DICE: {avg_val_loss_dict['dice']:.4f}, SegFocal: {avg_val_loss_dict['seg_focal']:.4f}, "
                      f"Focal: {avg_val_loss_dict['focal']:.4f}")
                print(f"  Val   - Swin Contrastive: {avg_val_loss_dict['swin_contrastive']:.4f}, "
                      f"ConvNeXt Coarse: {avg_val_loss_dict['convnext_coarse']:.4f}")
                print(f"  Val OF1: {overall_of1:.4f} (Authentic: {authentic_of1:.4f}, Forged: {forged_of1:.4f})")
                
                history['epochs'].append(epoch)
                history['train_loss_total'].append(avg_loss)
                history['train_loss_dice'].append(avg_loss_dict['dice'])
                history['train_loss_seg_focal'].append(avg_loss_dict['seg_focal'])
                history['train_loss_focal'].append(avg_loss_dict['focal'])
                history['train_loss_swin_contrastive'].append(avg_loss_dict['swin_contrastive'])
                history['train_loss_convnext_coarse'].append(avg_loss_dict['convnext_coarse'])
                history['val_loss_total'].append(avg_val_loss)
                history['val_loss_dice'].append(avg_val_loss_dict['dice'])
                history['val_loss_seg_focal'].append(avg_val_loss_dict['seg_focal'])
                history['val_loss_focal'].append(avg_val_loss_dict['focal'])
                history['val_loss_swin_contrastive'].append(avg_val_loss_dict['swin_contrastive'])
                history['val_loss_convnext_coarse'].append(avg_val_loss_dict['convnext_coarse'])
                history['val_of1_overall'].append(overall_of1)
                history['val_of1_authentic'].append(authentic_of1)
                history['val_of1_forged'].append(forged_of1)
                history['lr'].append(scheduler.get_last_lr()[0])
                
                # Model saving based ONLY on OF1
                is_best = overall_of1 > best_of1
                if is_best:
                    best_of1 = overall_of1
                    print(f"✨ New best overall OF1: {best_of1:.4f}")
                
                # Early stopping based ONLY on validation loss
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    epochs_without_improvement = 0
                    print(f"📉 Validation loss improved: {best_val_loss:.4f}")
                else:
                    epochs_without_improvement += 1
                    print(f"⏳ No val loss improvement for {epochs_without_improvement} validation(s)")
                
                save_checkpoint(
                    model, optimizer, scheduler, scaler, epoch, best_of1,
                    config.CHECKPOINT_DIR, is_best=is_best
                )
                
                history_path = os.path.join(config.OUTPUT_DIR, 'training_history.json')
                Path(config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
                with open(history_path, 'w') as f:
                    json.dump(history, f, indent=2)
                
                # Early Stopping Check
                if epochs_without_improvement >= config.EARLY_STOPPING_PATIENCE:
                    print(f"\n🛑 Early stopping triggered after {epochs_without_improvement} validations without improvement")
                    print(f"Best Validation Loss: {best_val_loss:.4f}, Best OF1: {best_of1:.4f}")
                    break
    
    if rank == 0:
        print(f"\n{'='*60}")
        print(f"Training Complete! Best OF1: {best_of1:.4f}")
        print(f"{'='*60}\n")
    
    cleanup_ddp()
    return history, best_of1

def main():
    """Main entry point"""
    import sys
    smoke_test = '--smoke-test' in sys.argv
    
    if smoke_test:
        print(f"🔥 SMOKE TEST MODE REQUESTED")
    
    world_size = Config.WORLD_SIZE
    
    import torch.multiprocessing as mp
    mp.spawn(train_ddp, args=(world_size, Config, smoke_test), nprocs=world_size, join=True)

if __name__ == '__main__':
    main()

Writing train.py


In [7]:
%%writefile test.py
"""
Testing and evaluation script with DDP support - Uses best thresholds from threshold.py
Save as: test.py
"""

import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import cv2
from sklearn.metrics import roc_auc_score, roc_curve
import json

from config import Config
from model import create_model
from dataset import ForgeryDataset, get_val_transform
from utils import oF1_score


def setup_ddp(rank, world_size):
    """Initialize DDP"""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12356'
    dist.init_process_group(Config.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)


def cleanup_ddp():
    """Cleanup DDP"""
    dist.destroy_process_group()


def load_best_model(config, device):
    """Load best checkpoint"""
    best_checkpoint_path = os.path.join(config.CHECKPOINT_DIR, 'best_checkpoint.pth')
    current_checkpoint_path = os.path.join(config.CHECKPOINT_DIR, 'current_checkpoint.pth')
    
    if os.path.exists(best_checkpoint_path):
        checkpoint_path = best_checkpoint_path
        if device == 0:
            print(f"✅ Loading BEST checkpoint")
    elif os.path.exists(current_checkpoint_path):
        checkpoint_path = current_checkpoint_path
        if device == 0:
            print(f"⚠️  Best checkpoint not found, loading CURRENT checkpoint")
    else:
        raise FileNotFoundError(f"No checkpoint found in {config.CHECKPOINT_DIR}")
    
    model = create_model(config).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=f'cuda:{device}', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    if device == 0:
        print(f"   Epoch: {checkpoint['epoch']}")
        print(f"   Best OF1 seen: {checkpoint.get('best_of1', 'N/A')}")
    
    return model


def load_best_thresholds(config):
    """Load best thresholds from threshold search results"""
    threshold_results_path = os.path.join(config.OUTPUT_DIR, 'threshold_search_results.json')
    
    if os.path.exists(threshold_results_path):
        with open(threshold_results_path, 'r') as f:
            results = json.load(f)
        
        # Find best configuration
        best_config = None
        best_of1 = -1.0
        
        for key, metrics in results.items():
            if metrics['overall'] > best_of1:
                best_of1 = metrics['overall']
                active_thresh, mask_thresh = map(float, key.split('_'))
                best_config = (active_thresh, mask_thresh)
        
        if best_config:
            print(f"✅ Loaded best thresholds from threshold search:")
            print(f"   Active Channel Threshold: {best_config[0]:.2f}")
            print(f"   Mask Threshold: {best_config[1]:.2f}")
            print(f"   Expected OF1: {best_of1:.4f}")
            return best_config[0], best_config[1]
    
    # Fallback to config defaults
    print(f"⚠️  No threshold search results found, using config defaults")
    return config.ACTIVE_CHANNEL_THRESHOLD, config.MASK_THRESHOLD


def evaluate_dataset_ddp(model, dataloader, config, device, rank, world_size, dataset_name="Dataset", 
                         save_worst=False, active_thresh=None, mask_thresh=None):
    """Evaluate dataset with DDP support - Memory efficient version"""
    model.eval()
    
    # Use provided thresholds or fall back to config
    if active_thresh is None:
        active_thresh = config.ACTIVE_CHANNEL_THRESHOLD
    if mask_thresh is None:
        mask_thresh = config.MASK_THRESHOLD
    
    # Lightweight results for gathering
    all_scores = []
    
    # Only store full data for worst cases if requested
    worst_cases = [] if save_worst else None
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f"Evaluating {dataset_name}", disable=(rank != 0))
        for batch in pbar:
            inputs_224 = batch['input_224'].to(device)
            inputs_448 = batch['input_448'].to(device)
            gt_masks = batch['masks'].to(device)
            is_forged = batch['is_forged']
            image_names = batch['image_name']
            
            outputs = model(inputs_224, inputs_448)
            pred_masks = outputs['segmentation']
            active_logits = outputs['active_logits']
            
            active_probs = torch.sigmoid(active_logits)
            active_channels = (active_probs > active_thresh).cpu().numpy()
            
            batch_size = pred_masks.shape[0]
            for b in range(batch_size):
                active_indices = np.where(active_channels[b])[0]
                
                pred_masks_list = []
                for idx in active_indices:
                    mask = pred_masks[b, idx].cpu().numpy()
                    mask = (mask > mask_thresh).astype(np.float32)
                    pred_masks_list.append(mask)
                
                gt_masks_list = []
                for i in range(config.NUM_INSTANCES):
                    mask = gt_masks[b, i].cpu().numpy()
                    if mask.sum() > 0:
                        gt_masks_list.append(mask)
                
                if len(gt_masks_list) == 0:
                    score = 1.0 if len(pred_masks_list) == 0 else 0.0
                    predicted_forged = len(pred_masks_list) > 0
                else:
                    score = 0.0 if len(pred_masks_list) == 0 else oF1_score(pred_masks_list, gt_masks_list)
                    predicted_forged = len(pred_masks_list) > 0
                
                # Lightweight result
                all_scores.append({
                    'image_name': image_names[b],
                    'is_forged': is_forged[b].item(),
                    'predicted_forged': predicted_forged,
                    'of1': score,
                    'num_pred_masks': len(pred_masks_list),
                    'num_gt_masks': len(gt_masks_list)
                })
                
                # Save worst cases for visualization (only if requested and forged)
                if save_worst and is_forged[b].item() and len(gt_masks_list) > 0:
                    worst_cases.append({
                        'image_name': image_names[b],
                        'of1': score,
                        'pred_masks': pred_masks_list,
                        'gt_masks': gt_masks_list,
                        'input_image': inputs_224[b].cpu().numpy()
                    })
    
    # Gather lightweight scores from all GPUs
    all_scores_gathered = [None] * world_size
    dist.all_gather_object(all_scores_gathered, all_scores)
    
    if rank == 0:
        # Flatten results from all GPUs
        all_results = []
        for scores in all_scores_gathered:
            all_results.extend(scores)
        
        # Calculate metrics
        authentic_results = [r for r in all_results if not r['is_forged']]
        forged_results = [r for r in all_results if r['is_forged']]
        
        authentic_of1 = np.mean([r['of1'] for r in authentic_results]) if authentic_results else 0.0
        forged_of1 = np.mean([r['of1'] for r in forged_results]) if forged_results else 0.0
        overall_of1 = np.mean([r['of1'] for r in all_results]) if all_results else 0.0
        
        # ROC AUC
        y_true = [int(r['is_forged']) for r in all_results]
        y_pred = [int(r['predicted_forged']) for r in all_results]
        
        if len(set(y_true)) > 1:
            roc_auc = roc_auc_score(y_true, y_pred)
        else:
            roc_auc = 0.0
        
        metrics = {
            'overall_of1': overall_of1,
            'authentic_of1': authentic_of1,
            'forged_of1': forged_of1,
            'roc_auc': roc_auc,
            'active_threshold': active_thresh,
            'mask_threshold': mask_thresh
        }
        
        # Process worst cases if available
        worst_results = None
        if save_worst:
            # Gather worst cases from all GPUs (only if they exist)
            all_worst_gathered = [None] * world_size
            if worst_cases is None:
                worst_cases = []
            dist.all_gather_object(all_worst_gathered, worst_cases)
            
            # Flatten and sort
            all_worst = []
            for cases in all_worst_gathered:
                if cases:
                    all_worst.extend(cases)
            
            if all_worst:
                # Sort by OF1 and take top N worst
                all_worst_sorted = sorted(all_worst, key=lambda x: x['of1'])
                worst_results = all_worst_sorted[:config.SAVE_TOP_WORST]
        
        return all_results, metrics, worst_results
    else:
        # Non-rank0 processes need to participate in gathering
        if save_worst:
            all_worst_gathered = [None] * world_size
            if worst_cases is None:
                worst_cases = []
            dist.all_gather_object(all_worst_gathered, worst_cases)
        return None, None, None


def visualize_worst_results(worst_results, output_dir, config):
    """Save visualizations of worst OF1 images"""
    vis_dir = os.path.join(output_dir, 'worst_predictions')
    Path(vis_dir).mkdir(parents=True, exist_ok=True)
    
    colors = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255),
        (255, 255, 0), (255, 0, 255), (0, 255, 255)
    ]
    
    for idx, result in enumerate(tqdm(worst_results, desc="Saving worst predictions")):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Original image
        img = result['input_image'].transpose(1, 2, 0)
        img = (img * 255).astype(np.uint8)
        axes[0].imshow(img)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # Ground truth masks
        gt_overlay = img.copy()
        for i, mask in enumerate(result['gt_masks']):
            mask_colored = np.zeros_like(img)
            color = colors[i % len(colors)]
            for c in range(3):
                mask_colored[:, :, c] = mask * color[c]
            gt_overlay = cv2.addWeighted(gt_overlay, 1.0, mask_colored.astype(np.uint8), 0.5, 0)
        
        axes[1].imshow(gt_overlay)
        axes[1].set_title(f'Ground Truth ({len(result["gt_masks"])} instances)')
        axes[1].axis('off')
        
        # Predicted masks
        pred_overlay = img.copy()
        for i, mask in enumerate(result['pred_masks']):
            mask_colored = np.zeros_like(img)
            color = colors[i % len(colors)]
            for c in range(3):
                mask_colored[:, :, c] = mask * color[c]
            pred_overlay = cv2.addWeighted(pred_overlay, 1.0, mask_colored.astype(np.uint8), 0.5, 0)
        
        axes[2].imshow(pred_overlay)
        axes[2].set_title(f'Predicted ({len(result["pred_masks"])} instances)\nOF1: {result["of1"]:.4f}')
        axes[2].axis('off')
        
        plt.suptitle(f'{result["image_name"]}', fontsize=10)
        plt.tight_layout()
        
        save_path = os.path.join(vis_dir, f'worst_{idx+1:03d}_{result["image_name"]}.png')
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        plt.close()


def plot_training_history(output_dir):
    """Plot all training curves"""
    history_path = os.path.join(output_dir, 'training_history.json')
    
    if not os.path.exists(history_path):
        print("⚠️  Training history not found")
        return
    
    with open(history_path, 'r') as f:
        history = json.load(f)
    
    has_epochs = 'epochs' in history and len(history['epochs']) > 0
    if not has_epochs:
        print("⚠️  No epoch data in history")
        return
    
    epochs = history['epochs']
    
    # Create 3x2 subplot
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    
    # Plot 1: Total Loss
    if 'train_loss_total' in history:
        axes[0, 0].plot(epochs, history['train_loss_total'], marker='o', color='blue', linewidth=2)
        axes[0, 0].set_xlabel('Epoch', fontsize=12)
        axes[0, 0].set_ylabel('Loss', fontsize=12)
        axes[0, 0].set_title('Total Training Loss', fontsize=14, fontweight='bold')
        axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Main Decoder Losses
    if all(k in history for k in ['train_loss_dice', 'train_loss_seg_focal', 'train_loss_focal']):
        axes[0, 1].plot(epochs, history['train_loss_dice'], label='DICE', marker='o', linewidth=2)
        axes[0, 1].plot(epochs, history['train_loss_seg_focal'], label='Seg Focal', marker='d', linewidth=2)
        axes[0, 1].plot(epochs, history['train_loss_focal'], label='Focal', marker='^', linewidth=2)
        axes[0, 1].set_xlabel('Epoch', fontsize=12)
        axes[0, 1].set_ylabel('Loss', fontsize=12)
        axes[0, 1].set_title('Main Decoder Losses', fontsize=14, fontweight='bold')
        axes[0, 1].legend(fontsize=10)
        axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Auxiliary Losses
    if all(k in history for k in ['train_loss_swin_contrastive', 'train_loss_convnext_coarse']):
        axes[1, 0].plot(epochs, history['train_loss_swin_contrastive'], 
                       label='Swin Contrastive', marker='s', color='green', linewidth=2)
        axes[1, 0].plot(epochs, history['train_loss_convnext_coarse'], 
                       label='ConvNeXt Coarse', marker='p', color='red', linewidth=2)
        axes[1, 0].set_xlabel('Epoch', fontsize=12)
        axes[1, 0].set_ylabel('Loss', fontsize=12)
        axes[1, 0].set_title('Auxiliary Head Losses', fontsize=14, fontweight='bold')
        axes[1, 0].legend(fontsize=10)
        axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Overall OF1
    if 'val_of1_overall' in history:
        axes[1, 1].plot(epochs, history['val_of1_overall'], 
                       label='Overall OF1', marker='o', color='purple', linewidth=2, markersize=8)
        axes[1, 1].set_xlabel('Epoch', fontsize=12)
        axes[1, 1].set_ylabel('OF1 Score', fontsize=12)
        axes[1, 1].set_title('Validation OF1 Score', fontsize=14, fontweight='bold')
        axes[1, 1].legend(fontsize=10)
        axes[1, 1].grid(True, alpha=0.3)
        
        best_idx = np.argmax(history['val_of1_overall'])
        best_of1 = history['val_of1_overall'][best_idx]
        best_epoch = epochs[best_idx]
        axes[1, 1].scatter([best_epoch], [best_of1], color='red', s=200, marker='*', 
                          zorder=5, label=f'Best: {best_of1:.4f}')
        axes[1, 1].legend(fontsize=10)
    
    # Plot 5: Authentic vs Forged OF1
    if 'val_of1_authentic' in history and 'val_of1_forged' in history:
        axes[2, 0].plot(epochs, history['val_of1_authentic'], 
                       label='Authentic OF1', marker='o', color='green', linewidth=2)
        axes[2, 0].plot(epochs, history['val_of1_forged'], 
                       label='Forged OF1', marker='s', color='red', linewidth=2)
        axes[2, 0].set_xlabel('Epoch', fontsize=12)
        axes[2, 0].set_ylabel('OF1 Score', fontsize=12)
        axes[2, 0].set_title('Authentic vs Forged OF1', fontsize=14, fontweight='bold')
        axes[2, 0].legend(fontsize=10)
        axes[2, 0].grid(True, alpha=0.3)
    
    # Plot 6: Learning Rate
    if 'lr' in history:
        axes[2, 1].plot(epochs, history['lr'], marker='o', color='brown', linewidth=2)
        axes[2, 1].set_xlabel('Epoch', fontsize=12)
        axes[2, 1].set_ylabel('Learning Rate', fontsize=12)
        axes[2, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
        axes[2, 1].grid(True, alpha=0.3)
        axes[2, 1].set_yscale('log')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Saved comprehensive training curves ({len(epochs)} epochs)")


def plot_roc_curve(all_results, output_dir, dataset_name):
    """Plot ROC curve"""
    if len(all_results) == 0:
        print("⚠️  No results to plot ROC curve")
        return
    
    y_true = [int(r['is_forged']) for r in all_results]
    y_pred = [int(r['predicted_forged']) for r in all_results]
    
    if len(set(y_true)) <= 1:
        print("⚠️  Cannot compute ROC curve (only one class present)")
        return
    
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {dataset_name}')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    
    plt.savefig(os.path.join(output_dir, f'roc_curve_{dataset_name.lower()}.png'), dpi=150)
    plt.close()
    
    print(f"✅ Saved ROC curve for {dataset_name} (AUC: {roc_auc:.4f})")


def test_ddp(rank, world_size, config):
    """Main testing function with DDP"""
    setup_ddp(rank, world_size)
    device = rank
    
    if rank == 0:
        print(f"\n{'='*60}")
        print(f"TESTING AND EVALUATION (DDP on {world_size} GPUs)")
        print(f"{'='*60}\n")
    
    # Load model
    model = load_best_model(config, device)
    model = DDP(model, device_ids=[rank], find_unused_parameters=True)
    
    # Load best thresholds
    active_thresh, mask_thresh = None, None
    if rank == 0:
        active_thresh, mask_thresh = load_best_thresholds(config)
    
    # Broadcast thresholds to all ranks
    threshold_obj = [[active_thresh, mask_thresh]]
    dist.broadcast_object_list(threshold_obj, src=0)
    active_thresh, mask_thresh = threshold_obj[0]
    
    # Create datasets
    if rank == 0:
        print("\n📦 Loading test dataset...")
    test_dataset = ForgeryDataset(
        authentic_dir=config.TEST_AUTHENTIC_DIR,
        forged_dir=config.TEST_FORGED_DIR,
        mask_dir=config.TEST_MASK_DIR,
        transform=get_val_transform(config),
        config=config
    )
    
    if rank == 0:
        print("\n📦 Loading HQ dataset...")
    hq_dataset = ForgeryDataset(
        authentic_dir=config.HQ_AUTHENTIC_DIR,
        forged_dir=config.HQ_FORGED_DIR,
        mask_dir=config.HQ_MASK_DIR,
        transform=get_val_transform(config),
        config=config
    )
    
    # Create distributed samplers and dataloaders
    test_sampler = DistributedSampler(test_dataset, num_replicas=world_size, rank=rank, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, sampler=test_sampler, 
                             num_workers=4, pin_memory=True)
    
    hq_sampler = DistributedSampler(hq_dataset, num_replicas=world_size, rank=rank, shuffle=False)
    hq_loader = DataLoader(hq_dataset, batch_size=config.BATCH_SIZE, sampler=hq_sampler,
                           num_workers=4, pin_memory=True)
    
    # Evaluate test dataset
    if rank == 0:
        print(f"\n🧪 Evaluating on test dataset with thresholds (A:{active_thresh:.2f}, M:{mask_thresh:.2f})...")
    test_results, test_metrics, worst_results = evaluate_dataset_ddp(
        model, test_loader, config, device, rank, world_size, "Test Dataset", 
        save_worst=True, active_thresh=active_thresh, mask_thresh=mask_thresh
    )
    
    # Evaluate HQ dataset (no worst case saving to save memory)
    if rank == 0:
        print(f"\n🧪 Evaluating on HQ dataset with thresholds (A:{active_thresh:.2f}, M:{mask_thresh:.2f})...")
    hq_results, hq_metrics, _ = evaluate_dataset_ddp(
        model, hq_loader, config, device, rank, world_size, "HQ Dataset", 
        save_worst=False, active_thresh=active_thresh, mask_thresh=mask_thresh
    )
    
    # Only rank 0 does the rest
    if rank == 0:
        # Print metrics
        print(f"\n{'='*60}")
        print("TEST DATASET RESULTS")
        print(f"{'='*60}")
        print(f"Active Threshold: {test_metrics['active_threshold']:.2f}")
        print(f"Mask Threshold:   {test_metrics['mask_threshold']:.2f}")
        print(f"Overall OF1:      {test_metrics['overall_of1']:.4f}")
        print(f"Authentic OF1:    {test_metrics['authentic_of1']:.4f}")
        print(f"Forged OF1:       {test_metrics['forged_of1']:.4f}")
        print(f"ROC AUC:          {test_metrics['roc_auc']:.4f}")
        
        print(f"\n{'='*60}")
        print("HQ DATASET RESULTS")
        print(f"{'='*60}")
        print(f"Active Threshold: {hq_metrics['active_threshold']:.2f}")
        print(f"Mask Threshold:   {hq_metrics['mask_threshold']:.2f}")
        print(f"Overall OF1:      {hq_metrics['overall_of1']:.4f}")
        print(f"Authentic OF1:    {hq_metrics['authentic_of1']:.4f}")
        print(f"Forged OF1:       {hq_metrics['forged_of1']:.4f}")
        print(f"ROC AUC:          {hq_metrics['roc_auc']:.4f}")
        
        # Save metrics
        Path(config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        
        all_metrics = {
            'test': test_metrics,
            'hq': hq_metrics
        }
        
        with open(os.path.join(config.OUTPUT_DIR, 'final_metrics.json'), 'w') as f:
            json.dump(all_metrics, f, indent=2)
        
        print(f"\n✅ Saved metrics to {config.OUTPUT_DIR}/final_metrics.json")
        
        # Visualizations
        print("\n📊 Generating visualizations...")
        
        # Plot training history
        plot_training_history(config.OUTPUT_DIR)
        
        # Plot ROC curves
        plot_roc_curve(test_results, config.OUTPUT_DIR, "Test")
        plot_roc_curve(hq_results, config.OUTPUT_DIR, "HQ")
        
        # Visualize worst predictions
        if worst_results:
            print(f"\n🖼️  Saving worst {len(worst_results)} predictions...")
            visualize_worst_results(worst_results, config.OUTPUT_DIR, config)
        
        print(f"\n{'='*60}")
        print("TESTING COMPLETE!")
        print(f"{'='*60}\n")
    
    cleanup_ddp()


def main():
    """Main entry point"""
    world_size = Config.WORLD_SIZE
    
    import torch.multiprocessing as mp
    mp.spawn(test_ddp, args=(world_size, Config), nprocs=world_size, join=True)


if __name__ == '__main__':
    main()

Writing test.py


In [8]:
%%writefile threshold.py
"""
Threshold Grid Search Script - Optimized for T4 x 2 DDP
"""

import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Subset
from torch.utils.data.distributed import DistributedSampler
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import json
import random
import torch.multiprocessing as mp

from config import Config
from model import create_model
from dataset import ForgeryDataset, get_val_transform
from utils import oF1_score

# --- OVERRIDE CONFIG FOR OPTIMIZATION ---
# T4 GPUs have 16GB VRAM. 32-64 is usually safe for 224/448px inputs.
# Adjust this if you get OOM (Out of Memory).
BATCH_SIZE_PER_GPU = 110
NUM_WORKERS = 4

def setup_ddp(rank, world_size):
    """Initialize DDP with NCCL backend (fastest for GPUs)"""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355' # Changed port to avoid conflicts
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp():
    dist.destroy_process_group()

def load_best_model(config, device):
    """Load best checkpoint safely"""
    # Prefer local best, fallback to kaggle input
    paths_to_check = [
        os.path.join(config.CHECKPOINT_DIR, 'best_checkpoint.pth'),
        "/kaggle/input/sci-forge-tr-model/pytorch/default/1/best_checkpoint.pth",
        os.path.join(config.CHECKPOINT_DIR, 'current_checkpoint.pth')
    ]
    
    checkpoint_path = None
    for p in paths_to_check:
        if os.path.exists(p):
            checkpoint_path = p
            break
            
    if not checkpoint_path:
        raise FileNotFoundError(f"No checkpoint found in {config.CHECKPOINT_DIR} or input dirs")

    if device == 0:
        print(f"✅ Loading checkpoint: {checkpoint_path}")

    model = create_model(config)
    # Move to specific GPU *before* DDP
    model = model.to(device)
    
    checkpoint = torch.load(checkpoint_path, map_location=f'cuda:{device}', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    return model

def get_sampled_indices(dataset, config):
    """
    Select indices for the subset.
    We do this ONCE on CPU so we can create a proper Subset.
    """
    all_indices = list(range(len(dataset)))
    
    authentic_indices = []
    forged_indices = []
    
    # Quick scan of dataset labels (assuming fast access)
    for idx in all_indices:
        if dataset.samples[idx]['is_forged']: # Optimized access if available
             forged_indices.append(idx)
        else:
             authentic_indices.append(idx)
    
    total_to_sample = int(len(all_indices) * config.VAL_SAMPLE_RATIO)
    n_authentic = int(total_to_sample * config.VAL_AUTHENTIC_RATIO)
    n_forged = total_to_sample - n_authentic
    
    n_authentic = min(n_authentic, len(authentic_indices))
    n_forged = min(n_forged, len(forged_indices))
    
    sampled_authentic = random.sample(authentic_indices, n_authentic) if n_authentic > 0 else []
    sampled_forged = random.sample(forged_indices, n_forged) if n_forged > 0 else []
    
    return list(set(sampled_authentic + sampled_forged))

def evaluate_with_threshold_ddp(model, dataloader, config, device, rank, active_thresh, mask_thresh):
    """
    Evaluate with specific thresholds.
    Note: 'sampled_indices' logic is removed here because the Dataloader 
    now ONLY contains the sampled data.
    """
    model.eval()
    
    authentic_scores = []
    forged_scores = []
    
    # Only show progress bar on Rank 0
    iterator = tqdm(dataloader, desc=f"Eval A={active_thresh:.2f} M={mask_thresh:.2f}", leave=False) if rank == 0 else dataloader

    with torch.no_grad():
        for batch in iterator:
            inputs_224 = batch['input_224'].to(device, non_blocking=True)
            inputs_448 = batch['input_448'].to(device, non_blocking=True)
            gt_masks = batch['masks'].to(device, non_blocking=True)
            is_forged = batch['is_forged'] # Keep on CPU for list appending
            
            outputs = model(inputs_224, inputs_448)
            pred_masks = outputs['segmentation']
            active_logits = outputs['active_logits']
            
            active_probs = torch.sigmoid(active_logits)
            
            # Move only necessary boolean array to CPU for logic
            active_channels_batch = (active_probs > active_thresh).cpu().numpy()
            
            batch_size = pred_masks.shape[0]
            
            for b in range(batch_size):
                active_indices = np.where(active_channels_batch[b])[0]
                
                pred_masks_list = []
                for idx in active_indices:
                    # Keep mask on GPU for thresholding if possible, but moving to numpy for oF1
                    mask = pred_masks[b, idx]
                    mask = (mask > mask_thresh).float().cpu().numpy()
                    pred_masks_list.append(mask)
                
                gt_masks_list = []
                # GT masks are usually sparse, process safely
                for i in range(config.NUM_INSTANCES):
                    mask = gt_masks[b, i]
                    if mask.sum() > 0:
                        gt_masks_list.append(mask.cpu().numpy())
                
                # Scoring
                if len(gt_masks_list) == 0:
                    score = 1.0 if len(pred_masks_list) == 0 else 0.0
                else:
                    score = 0.0 if len(pred_masks_list) == 0 else oF1_score(pred_masks_list, gt_masks_list)
                
                if is_forged[b].item():
                    forged_scores.append(score)
                else:
                    authentic_scores.append(score)

    return authentic_scores, forged_scores

def plot_threshold_heatmap(threshold_results, output_dir, config):
    """Plot 3x3 heatmap"""
    active_thresholds = config.TEST_ACTIVE_THRESHOLDS
    mask_thresholds = config.TEST_MASK_THRESHOLDS
    
    matrix = np.zeros((len(active_thresholds), len(mask_thresholds)))
    for i, active_t in enumerate(active_thresholds):
        for j, mask_t in enumerate(mask_thresholds):
            matrix[i, j] = threshold_results[(active_t, mask_t)]['overall']
    
    plt.figure(figsize=(10, 8))
    im = plt.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    
    plt.xticks(range(len(mask_thresholds)), [f"{t:.2f}" for t in mask_thresholds])
    plt.yticks(range(len(active_thresholds)), [f"{t:.2f}" for t in active_thresholds])
    plt.xlabel('Mask Threshold')
    plt.ylabel('Active Channel Threshold')
    plt.title('Overall OF1 Score')
    
    for i in range(len(active_thresholds)):
        for j in range(len(mask_thresholds)):
            plt.text(j, i, f'{matrix[i, j]:.3f}', ha="center", va="center", color="black", fontweight='bold')
    
    plt.colorbar(im, label='OF1 Score')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'threshold_heatmap.png'), dpi=150)
    plt.close()

def threshold_search_ddp(rank, world_size, config):
    """Main DDP Function"""
    setup_ddp(rank, world_size)
    
    # -- 1. Model Setup --
    model = load_best_model(config, rank)
    model = DDP(model, device_ids=[rank], output_device=rank) # Optimized DDP call
    
    # -- 2. Dataset Setup --
    if rank == 0:
        print("\n📦 initializing dataset...")
    
    full_dataset = ForgeryDataset(
        authentic_dir=config.TEST_AUTHENTIC_DIR,
        forged_dir=config.TEST_FORGED_DIR,
        mask_dir=config.TEST_MASK_DIR,
        transform=get_val_transform(config),
        config=config
    )
    
    # -- 3. Sampling Logic (Efficient) --
    # Calculate indices on Rank 0 and broadcast to ensure everyone uses same subset
    sampled_indices_obj = [None]
    if rank == 0:
        indices = get_sampled_indices(full_dataset, config)
        sampled_indices_obj = [indices]
        print(f"📊 Processed Subset: {len(indices)} images")
    
    dist.broadcast_object_list(sampled_indices_obj, src=0)
    sampled_indices = sampled_indices_obj[0]
    
    # Create a SUBSET. This is crucial for speed.
    # We only load what we need.
    test_subset = Subset(full_dataset, sampled_indices)
    
    # -- 4. Dataloader --
    # Sampler handles splitting the Subset across GPUs
    test_sampler = DistributedSampler(test_subset, num_replicas=world_size, rank=rank, shuffle=False)
    
    test_loader = DataLoader(
        test_subset, 
        batch_size=BATCH_SIZE_PER_GPU, # Uses the bigger batch size
        sampler=test_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True # Keeps workers alive for speed
    )

    if rank == 0:
        print(f"🚀 Running on {world_size} GPUs | Batch Size: {BATCH_SIZE_PER_GPU} per GPU")
    
    # -- 5. Grid Search --
    results = {}
    
    # Use tqdm for the outer loop on Rank 0
    outer_loop = config.TEST_ACTIVE_THRESHOLDS
    if rank == 0:
        print(f"\n🔍 Starting Grid Search...")
    
    for active_thresh in outer_loop:
        for mask_thresh in config.TEST_MASK_THRESHOLDS:
            
            # Run Evaluation
            local_auth_scores, local_forg_scores = evaluate_with_threshold_ddp(
                model, test_loader, config, rank, rank, active_thresh, mask_thresh
            )
            
            # Gather results from all GPUs
            all_auth = [None for _ in range(world_size)]
            all_forg = [None for _ in range(world_size)]
            
            dist.all_gather_object(all_auth, local_auth_scores)
            dist.all_gather_object(all_forg, local_forg_scores)
            
            if rank == 0:
                # Combine results
                final_auth = [s for sublist in all_auth for s in sublist]
                final_forg = [s for sublist in all_forg for s in sublist]
                
                auth_of1 = np.mean(final_auth) if final_auth else 0.0
                forg_of1 = np.mean(final_forg) if final_forg else 0.0
                overall_of1 = np.mean(final_auth + final_forg) if (final_auth + final_forg) else 0.0
                
                results[(active_thresh, mask_thresh)] = {
                    'overall': overall_of1,
                    'authentic': auth_of1,
                    'forged': forg_of1
                }
                print(f"   [A:{active_thresh:.2f} M:{mask_thresh:.2f}] -> OF1: {overall_of1:.4f}")

    # -- 6. Save & Plot (Rank 0 only) --
    if rank == 0:
        Path(config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        
        # Save JSON
        str_key_results = {f"{k[0]}_{k[1]}": v for k, v in results.items()}
        with open(os.path.join(config.OUTPUT_DIR, 'threshold_search_results.json'), 'w') as f:
            json.dump(str_key_results, f, indent=2)
            
        # Plot
        plot_threshold_heatmap(results, config.OUTPUT_DIR, config)
        
        # Best Config
        best_cfg = max(results.items(), key=lambda x: x[1]['overall'])
        print(f"\n🏆 BEST: Active={best_cfg[0][0]:.2f}, Mask={best_cfg[0][1]:.2f}, OF1={best_cfg[1]['overall']:.4f}")

    cleanup_ddp()

def main():
    # Automatically detect GPU count
    n_gpus = torch.cuda.device_count()
    if n_gpus < 1:
        print("❌ No GPUs found!")
        return
        
    print(f"🔄 Spawning {n_gpus} processes...")
    mp.spawn(threshold_search_ddp, args=(n_gpus, Config), nprocs=n_gpus, join=True)

if __name__ == '__main__':
    main()

Writing threshold.py


In [9]:
# !python train.py --smoke-test

In [10]:
!python train.py

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [11]:
!python threshold.py

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [12]:
!python test.py

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [13]:

# import json
# from IPython.display import Image, display
# import os

# # Display metrics
# with open('/kaggle/working/outputs/final_metrics.json', 'r') as f:
#     metrics = json.load(f)

# print("\n" + "="*60)
# print("FINAL RESULTS")
# print("="*60)
# print("\nTest Dataset:")
# print(f"  Overall OF1:    {metrics['test']['overall_of1']:.4f}")
# print(f"  Authentic OF1:  {metrics['test']['authentic_of1']:.4f}")
# print(f"  Forged OF1:     {metrics['test']['forged_of1']:.4f}")
# print(f"  ROC AUC:        {metrics['test']['roc_auc']:.4f}")

# print("\nHQ Dataset:")
# print(f"  Overall OF1:    {metrics['hq']['overall_of1']:.4f}")
# print(f"  Authentic OF1:  {metrics['hq']['authentic_of1']:.4f}")
# print(f"  Forged OF1:     {metrics['hq']['forged_of1']:.4f}")
# print(f"  ROC AUC:        {metrics['hq']['roc_auc']:.4f}")
# print("="*60 + "\n")

# # Display training curves
# print("\n📊 Training Curves:")
# display(Image('/kaggle/working/outputs/training_curves.png'))

# print("\n📊 ROC Curve:")
# display(Image('/kaggle/working/outputs/roc_curve.png'))

# # Display some worst predictions
# print("\n🖼️  Worst Predictions (first 5):")
# worst_dir = '/kaggle/working/outputs/worst_predictions'
# worst_files = sorted([f for f in os.listdir(worst_dir) if f.endswith('.png')])[:5]

# for f in worst_files:
#     print(f"\n{f}")
#     display(Image(os.path.join(worst_dir, f)))

In [14]:
# import torch
# import matplotlib.pyplot as plt
# import numpy as np
# import cv2
# from torch.utils.data import DataLoader
# from config import Config
# from dataset import ForgeryDataset, get_train_transform

# def debug_dataloader_instances():
#     print("🔍 Starting Multi-Instance Mask Debugger...")
    
#     # 1. Config Setup
#     conf = Config()
#     conf.SMOKE_TEST = True
#     conf.SMOKE_TEST_SAMPLES = 8  # Keep it small for debugging
    

#     # 2. Initialize Dataset
#     # We use get_train_transform to ensure augmentations (like flips) 
#     # are applied consistently to both images and all mask instances.
#     ds = ForgeryDataset(
#         authentic_dir=conf.TRAIN_AUTHENTIC_DIR,
#         forged_dir=conf.TRAIN_FORGED_DIR,
#         mask_dir=conf.TRAIN_MASK_DIR,
#         transform=get_train_transform(conf),
#         config=conf
#     )
    
#     # 3. Load Batch
#     batch_size = 4
#     loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    
#     try:
#         batch = next(iter(loader))
#     except Exception as e:
#         print(f"❌ Error loading batch: {e}")
#         return

#     inputs = batch['input']           # Shape: (B, 5, H, W)
#     masks = batch['masks']            # Shape: (B, NUM_INSTANCES, H, W)
#     is_forged = batch['is_forged']
#     num_instances = batch['num_instances']
#     image_names = batch['image_name']
    
#     # 4. Visualize
#     # We iterate through the batch items
#     for i in range(min(batch_size, len(inputs))):
        
#         # Determine number of subplots: 1 (Image) + NUM_INSTANCES (Masks)
#         n_cols = 1 + conf.NUM_INSTANCES
#         fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3.5))
        
#         # --- 1. Input Image (RGB only) ---
#         # Inputs are (C, H, W). Channels 0,1,2 are RGB. 3,4 are X,Y (ignored here).
#         img_tensor = inputs[i, :3] 
#         img = img_tensor.permute(1, 2, 0).numpy()
#         img = np.clip(img, 0, 1) # Ensure valid range for plotting
        
#         # Title info
#         status = "FORGED" if is_forged[i] > 0.5 else "AUTHENTIC"
#         n_inst = num_instances[i].item()
        
#         axes[0].imshow(img)
#         axes[0].set_title(f"{status}\n{image_names[i]}\nFound {n_inst} masks", fontsize=9, color='red' if n_inst>0 else 'green')
#         axes[0].axis('off')
        
#         # --- 2. Mask Instances ---
#         # Masks tensor shape: (NUM_INSTANCES, H, W)
#         instance_masks = masks[i]
        
#         for idx in range(conf.NUM_INSTANCES):
#             ax = axes[idx + 1]
            
#             # Get specific instance mask
#             m = instance_masks[idx].numpy()
            
#             # Check if this mask instance has content
#             has_content = m.max() > 0
            
#             # visual styling
#             cmap = 'hot' if has_content else 'gray'
#             title_color = 'red' if has_content else 'black'
#             title_text = f"Instance {idx+1}" + (" (Active)" if has_content else "")
            
#             ax.imshow(m, cmap=cmap, vmin=0, vmax=1)
#             ax.set_title(title_text, fontsize=9, color=title_color)
#             ax.axis('off')
            
#             # Optional: Add a border to active masks for visibility
#             if has_content:
#                 for spine in ax.spines.values():
#                     spine.set_edgecolor('red')
#                     spine.set_linewidth(2)

#         plt.tight_layout()
#         plt.show()

# if __name__ == "__main__":
#     debug_dataloader_instances()